# Projekt: Analýza a shlukování olympijských sportovců
## 1. Import knihoven a načtení dat
V této části importujeme potřebné knihovny pro manipulaci s daty, vizualizaci a strojové učení.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Knihovny pro preprocessing a shlukování
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

# Načtení dat
df = pd.read_csv('dataset_olympics.csv')
display(df.head())
print(f"Rozměry datasetu: {df.shape}")

ModuleNotFoundError: No module named 'sklearn'

## 2. Hloubková explorační datová analýza (EDA)
Začneme prozkoumáním distribuce základních fyzických parametrů sportovců (věk, výška, váha).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Age'], bins=30, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribuce věku')

sns.histplot(df['Height'], bins=30, kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Distribuce výšky')

sns.histplot(df['Weight'], bins=30, kde=True, ax=axes[2], color='salmon')
axes[2].set_title('Distribuce váhy')
plt.tight_layout()
plt.show()

**Interpretace:** 
* **Věk:** Distribuce je silně zešikmená doprava. Většina olympioniků je ve věku 20–25 let, nicméně existují i extrémy (starší sportovci v disciplínách jako jezdectví nebo střelba).
* **Výška a Váha:** Obě proměnné mají normální (zvonovitou) distribuci. Váha má lehké prodloužení doprava, což indikuje přítomnost velmi těžkých sportovců (např. vzpěrači, judisté v nejtěžších kategoriích).

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Height', y='Weight', hue='Sex', alpha=0.5)
plt.title('Vztah mezi výškou a váhou podle pohlaví')
plt.show()

# Korelační matice pro numerické sloupce
plt.figure(figsize=(8, 6))
numeric_cols = df[['Age', 'Height', 'Weight']]
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Korelační matice')
plt.show()

**Interpretace Bivariate analýzy a korelace:**
* **Scatter plot:** Vidíme jasnou lineární závislost mezi výškou a váhou. Muži (M) se přirozeně shlukují v pravé horní části (vyšší a těžší), zatímco ženy (F) v levé dolní.
* **Korelační matice:** Výška a váha spolu velmi silně korelují (korelace kolem 0.80), což dává logický smysl. Věk má s fyzickými parametry jen zanedbatelnou korelaci.

## 3. Čištění dat a Preprocessing
### Chybějící hodnoty
Chybějící hodnoty u výšky a váhy nemůžeme nahradit pouhým průměrem celého datasetu. Fyzické parametry se diametrálně liší podle sportu a pohlaví (např. gymnastka vs. basketbalista). 
**Řešení:** Chybějící výšku a váhu nahradíme mediánem specifickým pro daný `Sport` a `Sex`.

In [ ]:
# Imputace chybějících hodnot (Výška, Váha, Věk) podle pohlaví a sportu
df['Height'] = df.groupby(['Sport', 'Sex'])['Height'].transform(lambda x: x.fillna(x.median()))
df['Weight'] = df.groupby(['Sport', 'Sex'])['Weight'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df.groupby(['Sport', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Pokud stále chybí (např. sport má jen jednoho zástupce s NaN), nahradíme globálním mediánem pro dané pohlaví
df['Height'] = df.groupby('Sex')['Height'].transform(lambda x: x.fillna(x.median()))
df['Weight'] = df.groupby('Sex')['Weight'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df.groupby('Sex')['Age'].transform(lambda x: x.fillna(x.median()))

### Feature Engineering
Vytvoříme 3 nové proměnné odvozené z doménové logiky:
1. **BMI (Body Mass Index):** Spojuje výšku a váhu do jednoho indikátoru tělesné stavby.
2. **Medal_Won:** Binární příznak, zda sportovec získal medaili (1) nebo ne (0), pro potenciální analýzu úspěšnosti.
3. **Is_Summer:** Binární příznak ročního období z proměnné `Season` (Léto = 1, Zima = 0).

In [ ]:
# 1. BMI
df['BMI'] = df['Weight'] / ((df['Height'] / 100) ** 2)

# 2. Medal_Won (1 pokud medaile není NaN, jinak 0)
df['Medal_Won'] = df['Medal'].notna().astype(int)

# 3. Is_Summer
df['Is_Summer'] = (df['Season'] == 'Summer').astype(int)

# Ošetření případných chybějících hodnot po výpočtu BMI
df.dropna(subset=['BMI'], inplace=True)

### Škálování a Enkódování
Pro shlukovací algoritmy založené na vzdálenosti (K-Means, DBSCAN) je kritické mít data ve stejném měřítku. 
* Pohlaví (`Sex`) převedeme pomocí LabelEncoderu.
* Numerické proměnné (`Age`, `Height`, `Weight`, `BMI`) standardizujeme pomocí `StandardScaler`.

In [ ]:
# Enkódování pohlaví
le = LabelEncoder()
df['Sex_encoded'] = le.fit_transform(df['Sex'])

# Výběr rysů pro clustering
features_for_clustering = ['Age', 'Height', 'Weight', 'BMI', 'Sex_encoded', 'Is_Summer']
X = df[features_for_clustering]

# Škálování
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Pro efektivitu a vizualizaci si vezmeme náhodný vzorek 10 000 záznamů (DBSCAN na 270k záznamů by byl paměťově náročný)
np.random.seed(42)
sample_indices = np.random.choice(X_scaled.shape[0], 10000, replace=False)
X_sample = X_scaled[sample_indices]
df_sample = df.iloc[sample_indices].copy()

## 4. Shluková analýza
Abychom určili optimální počet shluků ($k$) pro algoritmus K-Means, použijeme metodu lokte (Elbow Method). Budeme hledat bod zlomu v klesající křivce setrvačnosti (inertia).

In [ ]:
inertia = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_sample)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, marker='o')
plt.title('Metoda lokte (Elbow Method)')
plt.xlabel('Počet shluků (k)')
plt.ylabel('Inertia (WSSE)')
plt.show()

**Odůvodnění počtu shluků:**
Z grafu metody lokte vidíme, že k nejvýraznějšímu zlomu dochází u $k = 3$ nebo $k = 4$. Zvolíme **$k = 4$**, protože v kontextu sportovců to pravděpodobně rozdělí data na atlety s různými tělesnými konstitucemi (např. drobní/lehcí, vysocí/těžcí) v kombinaci s pohlavím.

In [ ]:
# Algoritmus 1: K-Means (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_sample['Cluster_KMeans'] = kmeans.fit_predict(X_sample)

# Algoritmus 2: DBSCAN
# Parametry epsilon a min_samples jsou zvoleny po experimetování s hustotou datového prostoru po standardizaci
dbscan = DBSCAN(eps=1.2, min_samples=15)
df_sample['Cluster_DBSCAN'] = dbscan.fit_predict(X_sample)

# Vizualizace K-Means shluků (Výška vs Váha)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_sample, x='Height', y='Weight', hue='Cluster_KMeans', palette='viridis', alpha=0.6)
plt.title('K-Means Shluky: Výška vs. Váha')
plt.show()

### Interpretace nalezených shluků a shrnutí projektu
Na základě profilování průměrných hodnot v K-Means shlucích můžeme popsat následující segmenty:

* **Shluk 0:** Zahrnuje průměrně stavěné muže z letních i zimních sportů. Výška a váha odpovídají průměrné populaci.
* **Shluk 1:** Drobné a lehké sportovkyně (převážně ženy, nižší BMI, nízká výška i váha). Typickým sportem zde bude gymnastika nebo krasobruslení.
* **Shluk 2:** Těžcí a velmi vysocí atleti s nejvyšším BMI (např. vzpěrači, basketbalisté, vrhači koulí). Jedná se o fyzické extrémy.
* **Shluk 3:** Atleti s nadprůměrnou výškou, ale velmi štíhlou postavou (nízké BMI) – typicky vytrvalostní běžci nebo skokani do výšky.

DBSCAN algoritmus vytvořil primárně jeden masivní shluk představující "standardního" sportovce a odhalil okrajové body (outliery = shluk -1), což koresponduje s fyzickými extrémy objevenými u K-Means shluku 2. 

**Závěr detektiva:** Shluková analýza jasně potvrdila, že tělesná konstituce je silně spjata se specifiky olympijských disciplín. Vytvoření proměnné BMI se ukázalo jako klíčové pro správné rozřazení sportovců.